# 02｜切換 Collection、無答案題、換生成模型

這本 notebook 補完企畫書 Stage 1～2 還沒驗證的三件事：

| 測試 | 要證明什麼 | 需要 LLM 嗎 |
|---|---|---|
| **隔離測試** | 用 A 的索引查詢，只會拿到 A 的文件；B 也一樣 | 不用 |
| **無答案題** | B 的題目拿去問 A（A 裡沒有答案），模型應該拒答，不能亂掰 | 要 |
| **換生成模型** | 換 1.7B／4B／9B 模型時，用的是同一份索引、沒有重建 | 要 |

| Collection | 內容 |
|---|---|
| A：`squad_v11_dev500` | 原本的 11 篇文章、534 段 |
| B：`squad_v11_b5` | 另外 5 篇主題不重疊的文章、185 段（Super Bowl 50、Prime number、Genghis Khan、Black Death、Packet switching） |

B 的 15 題已經自動排除「答案也出現在 A 裡」的題目（見 `eval/squad_b_manifest.json`）。

**執行時間：** 每個模型約 130 次呼叫（A 的 50 題 × RAG＋oracle、B 的 15 題 × 2 種），三個模型合計約 400 次。4B 每次約 2～3 秒，9B 會慢一些。可以先把 `N_A` 設成 5 試跑。

`src/rag_core.py` 是從 01 整理出來的同一套程式（索引、檢索、Prompt、解析、評分）。01 保持原樣，02 之後都用這個模組。

## 0. 設定

In [1]:
import json
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "collections").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src import rag_core as rc

COLL_A, COLL_B = "squad_v11_dev500", "squad_v11_b5"
Q_A_FILE, Q_B_FILE = "squad_dev.jsonl", "squad_b_questions.jsonl"

EMBED_MODEL = "BAAI/bge-small-en-v1.5"
OLLAMA_URL = "http://localhost:11434"
NUM_CTX = 4096
CONTEXT_CHAR_BUDGET = 8000
TOP_K = 3

# (模型名稱, think)：會「思考」的模型設 False 關掉思考，不會思考的設 None（不傳參數）
GEN_MODELS = [
    ("qwen3:1.7b", False),
    ("qwen3:4b-instruct", None),
    ("qwen3.5:9b-q4_K_M", False),
]
N_A = 50              # A 的題目用幾題；先用 5 試跑

# 填既有的 run_id 就直接讀紀錄畫表，不呼叫 LLM
LOAD_RUN = None

pd.set_option("display.max_colwidth", 100)
print("ROOT =", ROOT)

ROOT = <專案根目錄>


## 1. 載入兩個 Collection 與題目

In [2]:
docs_A, docs_B = rc.load_collection(ROOT, COLL_A), rc.load_collection(ROOT, COLL_B)
ids_A, ids_B = {d["doc_id"] for d in docs_A}, {d["doc_id"] for d in docs_B}
titles_A, titles_B = {d["title"] for d in docs_A}, {d["title"] for d in docs_B}

q_A = rc.load_jsonl(ROOT / "eval" / Q_A_FILE)[:N_A]
q_B = rc.load_jsonl(ROOT / "eval" / Q_B_FILE)

print(f"A：{len(docs_A)} 段、{len(titles_A)} 篇文章；題目 {len(q_A)} 題")
print(f"B：{len(docs_B)} 段、{len(titles_B)} 篇文章；題目 {len(q_B)} 題")
print("兩套共用的 doc_id：", ids_A & ids_B or "無")
print("兩套共用的文章：", titles_A & titles_B or "無")
assert not (ids_A & ids_B) and not (titles_A & titles_B)
assert all(q["gold_doc_id"] in ids_A for q in q_A) and all(q["gold_doc_id"] in ids_B for q in q_B)

A：534 段、11 篇文章；題目 50 題
B：185 段、5 篇文章；題目 15 題
兩套共用的 doc_id： 無
兩套共用的文章： 無


## 2. 建立或載入兩份索引

兩套文件各有自己的索引，索引名稱裡含有文件內容的指紋，所以不可能混用。A 的索引在 01 已經建過，這裡應該顯示「讀取快取」。

In [3]:
embedder = rc.Embedder(EMBED_MODEL)
idx_A = rc.build_or_load_index(ROOT, docs_A, COLL_A, embedder)
idx_B = rc.build_or_load_index(ROOT, docs_B, COLL_B, embedder)
ret_A = rc.DenseRetriever(idx_A, docs_A, embedder)
ret_B = rc.DenseRetriever(idx_B, docs_B, embedder)

pd.DataFrame([
    {"Collection": COLL_A, "索引": idx_A["key"], "這次": "新建" if idx_A["built"] else "讀取快取",
     "秒": idx_A["seconds"], "被截斷的段落": embedder.n_truncated([d["text"] for d in docs_A])},
    {"Collection": COLL_B, "索引": idx_B["key"], "這次": "新建" if idx_B["built"] else "讀取快取",
     "秒": idx_B["seconds"], "被截斷的段落": embedder.n_truncated([d["text"] for d in docs_B])},
])

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (735 > 512). Running this sequence through the model will result in indexing errors


,Collection,索引,這次,秒,被截斷的段落
0,squad_v11_dev500,squad_v11_dev500__BAAI-bge-small-en-v1-5__153e36d1e3b4,讀取快取,0.0,0
1,squad_v11_b5,squad_v11_b5__BAAI-bge-small-en-v1-5__f513c4c0ef20,新建,0.9,1


## 3. 隔離測試（不呼叫 LLM）

四種組合都取 Top-10，檢查有沒有出現另一套的文件。題目和索引同一套時，順便算 gold 段落的命中率。

In [4]:
def isolation(ret, own_ids, qs, label_idx, label_q):
    foreign, ranks = 0, []
    for q in qs:
        hits = ret.search(q["question"], 10)
        foreign += sum(h["doc_id"] not in own_ids for h in hits)
        if q["gold_doc_id"] in own_ids:
            ranks.append(ret.rank_of(q["question"], q["gold_doc_id"]))
    row = {"索引": label_idx, "題目": label_q, "題數": len(qs), "Top-10 中出現另一套的文件": foreign,
           "結果": "PASS" if foreign == 0 else "FAIL"}
    if ranks:
        r = np.array(ranks)
        row.update({"hit@1": round((r <= 1).mean(), 3), "hit@3": round((r <= 3).mean(), 3)})
    return row


iso = pd.DataFrame([
    isolation(ret_A, ids_A, q_A, "A", "A 的題目"),
    isolation(ret_A, ids_A, q_B, "A", "B 的題目（A 裡沒有答案）"),
    isolation(ret_B, ids_B, q_B, "B", "B 的題目"),
    isolation(ret_B, ids_B, q_A, "B", "A 的題目（B 裡沒有答案）"),
])
iso

,索引,題目,題數,Top-10 中出現另一套的文件,結果,hit@1,hit@3
0,A,A 的題目,50,0,PASS,0.660,0.880
1,A,B 的題目（A 裡沒有答案）,15,0,PASS,NaN,NaN
2,B,B 的題目,15,0,PASS,0.733,0.867
3,B,A 的題目（B 裡沒有答案）,50,0,PASS,NaN,NaN


### 相似度分數能不能用來判斷「沒有答案」？

比較「有答案的題目」和「沒答案的題目」在 A 索引上的 Top-1 分數。如果兩者範圍重疊，就沒辦法單靠設一個分數門檻來拒答，還是得靠模型判斷。

In [5]:
top1 = pd.DataFrame(
    [{"題目": "A 的題目（有答案）", "top1_score": ret_A.search(q["question"], 1)[0]["score"]} for q in q_A] +
    [{"題目": "B 的題目（沒答案）", "top1_score": ret_A.search(q["question"], 1)[0]["score"]} for q in q_B])
top1.groupby("題目").top1_score.describe()[["count", "min", "25%", "50%", "75%", "max"]].round(3)

,count,min,25%,50%,75%,max
題目,,,,,,
A 的題目（有答案）,50.0,0.583,0.684,0.730,0.766,0.861
B 的題目（沒答案）,15.0,0.498,0.533,0.559,0.579,0.618


## 4. 換生成模型＋無答案題（呼叫 LLM）

每個模型跑四組：

| 題組 | 用哪份索引 | 模式 | 正確行為 |
|---|---|---|---|
| `A_dev` | A | rag、oracle | 答對 |
| `B_on_A` | A | rag | **拒答**（A 裡沒有答案） |
| `B_on_B` | B | rag | 答對（證明切到 B 之後能正常運作） |

所有模型共用第 2 節建好的同一份索引物件，迴圈裡不會重新編碼文件。每筆紀錄都會寫下它用的 `index_key`，跑完再檢查。

In [6]:
def run_model(model, think):
    gen = rc.OllamaGenerator(model, OLLAMA_URL, NUM_CTX, think)
    gen.check()
    recs, t0 = [], time.time()
    gold_A = {d["doc_id"]: d for d in docs_A}
    jobs = ([("A_dev", q, "rag", ret_A) for q in q_A] + [("A_dev", q, "oracle", ret_A) for q in q_A] +
            [("B_on_A", q, "rag", ret_A) for q in q_B] + [("B_on_B", q, "rag", ret_B) for q in q_B])
    for i, (test_set, q, mode, ret) in enumerate(jobs, 1):
        r = rc.ask(q, mode, gen, ret, k=TOP_K, char_budget=CONTEXT_CHAR_BUDGET,
                   gold_doc=gold_A.get(q["gold_doc_id"]) if mode == "oracle" else None)
        r["test_set"] = test_set
        recs.append(r)
        if i % 25 == 0 or i == len(jobs):
            print(f"  {model}: {i}/{len(jobs)}，{time.time() - t0:.0f}s")
    return recs


emb_ids_before = (id(idx_A["emb"]), id(idx_B["emb"]))

if LOAD_RUN is None:
    run_id = datetime.now().strftime("%Y%m%d_%H%M%S") + "_collections_models"
    run_dir = ROOT / "results" / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "config.json").write_text(json.dumps({
        "run_id": run_id, "coll_A": COLL_A, "coll_B": COLL_B, "index_A": idx_A["key"], "index_B": idx_B["key"],
        "embed_model": EMBED_MODEL, "gen_models": GEN_MODELS, "top_k": TOP_K, "num_ctx": NUM_CTX,
        "context_char_budget": CONTEXT_CHAR_BUDGET, "n_A": len(q_A), "n_B": len(q_B),
        "system_rag": rc.SYSTEM_RAG}, indent=2, ensure_ascii=False), encoding="utf-8")
    records, skipped = [], []
    with (run_dir / "records.jsonl").open("w", encoding="utf-8") as f:
        for model, think in GEN_MODELS:
            try:
                recs = run_model(model, think)
            except RuntimeError as e:
                print(f"略過 {model}：{e}")
                skipped.append(model)
                continue
            records += recs
            for r in recs:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print("已存到", run_dir, "| 略過：", skipped or "無")
else:
    run_id, run_dir = LOAD_RUN, ROOT / "results" / LOAD_RUN
    records = rc.load_jsonl(run_dir / "records.jsonl")
    print(f"讀取 {run_id}：{len(records)} 筆")

assert (id(idx_A["emb"]), id(idx_B["emb"])) == emb_ids_before, "索引物件在迴圈中被換掉了"

  qwen3:1.7b: 25/130，86s
  qwen3:1.7b: 50/130，143s
  qwen3:1.7b: 75/130，200s
  qwen3:1.7b: 100/130，257s
  qwen3:1.7b: 125/130，316s
  qwen3:1.7b: 130/130，328s
  qwen3:4b-instruct: 25/130，78s
  qwen3:4b-instruct: 50/130，150s
  qwen3:4b-instruct: 75/130，215s
  qwen3:4b-instruct: 100/130，280s
  qwen3:4b-instruct: 125/130，346s
  qwen3:4b-instruct: 130/130，359s
  qwen3.5:9b-q4_K_M: 25/130，122s
  qwen3.5:9b-q4_K_M: 50/130，204s
  qwen3.5:9b-q4_K_M: 75/130，279s
  qwen3.5:9b-q4_K_M: 100/130，357s
  qwen3.5:9b-q4_K_M: 125/130，452s
  qwen3.5:9b-q4_K_M: 130/130，467s
已存到 <專案根目錄>\results\20261025_203003_collections_models | 略過： 無


## 5. 結果

### 5.1 換模型時有沒有重建索引

In [7]:
df = pd.DataFrame(records)
rag = df[df["mode"] == "rag"]
idx_check = (rag.groupby(["gen_model", "test_set"]).index_key.agg(lambda s: ", ".join(sorted(set(s))))
             .unstack())
print("A 的索引：", idx_A["key"], "\nB 的索引：", idx_B["key"])
print("所有模型 A_dev／B_on_A 都用 A 的索引：", set(rag[rag.test_set != "B_on_B"].index_key) == {idx_A["key"]})
print("所有模型 B_on_B 都用 B 的索引：", set(rag[rag.test_set == "B_on_B"].index_key) == {idx_B["key"]})
idx_check

A 的索引： squad_v11_dev500__BAAI-bge-small-en-v1-5__153e36d1e3b4 
B 的索引： squad_v11_b5__BAAI-bge-small-en-v1-5__f513c4c0ef20
所有模型 A_dev／B_on_A 都用 A 的索引： True
所有模型 B_on_B 都用 B 的索引： True


test_set,A_dev,B_on_A,B_on_B
gen_model,,,
qwen3.5:9b-q4_K_M,squad_v11_dev500__BAAI-bge-small-en-v1-5__153e36d1e3b4,squad_v11_dev500__BAAI-bge-small-en-v1-5__153e36d1e3b4,squad_v11_b5__BAAI-bge-small-en-v1-5__f513c4c0ef20
qwen3:1.7b,squad_v11_dev500__BAAI-bge-small-en-v1-5__153e36d1e3b4,squad_v11_dev500__BAAI-bge-small-en-v1-5__153e36d1e3b4,squad_v11_b5__BAAI-bge-small-en-v1-5__f513c4c0ef20
qwen3:4b-instruct,squad_v11_dev500__BAAI-bge-small-en-v1-5__153e36d1e3b4,squad_v11_dev500__BAAI-bge-small-en-v1-5__153e36d1e3b4,squad_v11_b5__BAAI-bge-small-en-v1-5__f513c4c0ef20


### 5.2 三個模型的比較

- **有答案的題目（A_dev）**：`答對率` 用 `contains_gold`（回答裡出現標準答案）；`誤拒答` 是明明有答案卻說證據不足。
- **沒答案的題目（B_on_A）**：`拒答率` 越高越好，`亂答` 是沒有答案卻給了答案。
- 單一次執行，題數又少，差 2～3 題可能只是隨機波動。

In [8]:
def summarize(g):
    a_rag = g[(g.test_set == "A_dev") & (g["mode"] == "rag")]
    a_ora = g[(g.test_set == "A_dev") & (g["mode"] == "oracle")]
    b_on_a = g[g.test_set == "B_on_A"]
    b_on_b = g[g.test_set == "B_on_B"]
    return pd.Series({
        "A_dev 答對率（RAG）": a_rag.contains_gold.mean(),
        "A_dev 答對率（oracle）": a_ora.contains_gold.mean(),
        "A_dev f1_qstrip（RAG）": a_rag.f1_qstrip.mean(),
        "A_dev 誤拒答": a_rag.abstained.mean(),
        "B_on_A 拒答率 ↑": b_on_a.abstained.mean(),
        "B_on_A 亂答題數 ↓": int((~b_on_a.abstained.astype(bool)).sum()),
        "B_on_B 答對率": b_on_b.contains_gold.mean(),
        "有寫 Sources": g[g["mode"] != "no_rag"].has_sources.mean(),
        "引用不在 Context": g.invalid_citations.map(bool).mean(),
        "平均秒數": g.latency_s.mean(),
        "平均輸出 tokens": g.output_tokens.mean(),
    })


order = [m for m, _ in GEN_MODELS if m in set(df.gen_model)]
summary = pd.DataFrame({m: summarize(df[df.gen_model == m]) for m in order}).round(2)
summary

,qwen3:1.7b,qwen3:4b-instruct,qwen3.5:9b-q4_K_M
A_dev 答對率（RAG）,0.60,0.82,0.82
A_dev 答對率（oracle）,0.60,0.92,0.92
A_dev f1_qstrip（RAG）,0.69,0.73,0.79
A_dev 誤拒答,0.02,0.08,0.08
B_on_A 拒答率 ↑,0.47,1.00,1.00
B_on_A 亂答題數 ↓,8.00,0.00,0.00
B_on_B 答對率,0.73,0.80,0.87
有寫 Sources,0.96,0.95,1.00
引用不在 Context,0.07,0.00,0.37
平均秒數,2.51,2.75,3.58


### 5.3 沒有答案卻亂答的案例

看模型把哪段文件當成依據。這些案例是 Stage 3 加入 BM25 之後要特別注意的副作用：關鍵字相同但主題無關的段落，可能讓這類亂答變多。

In [9]:
halluc = df[(df.test_set == "B_on_A") & (~df.abstained.astype(bool))].copy()
doc_title = {d["doc_id"]: d["title"] for d in docs_A}
halluc["拿到的段落（文章）"] = halluc.context_ids.map(lambda ids: ", ".join(sorted({doc_title[i] for i in ids})))
print(f"共 {len(halluc)} 筆")
halluc[["gen_model", "question", "gold_answers", "answer", "cited_ids", "拿到的段落（文章）"]]

共 8 筆


,gen_model,question,gold_answers,answer,cited_ids,拿到的段落（文章）
100,qwen3:1.7b,What did DECnet originally do,[connect two PDP-11 minicomputers],Provided a protocol for networking,[Oxygen__p002],"American Broadcasting Company, Oxygen"
103,qwen3:1.7b,"British commentators include, Darren Fletcher, Rocky Boiman and who else?",[Greg Brady],Alick Horsnell,[Victoria_and_Albert_Museum__p018],Victoria and Albert Museum
104,qwen3:1.7b,Who recovered the strip ball?,"[wards, Ward]",<unknown>,[],"Nikola Tesla, University of Chicago"
105,qwen3:1.7b,Which descendant of Genghis Khan is remembered as having reunified China?,[Kublai Khan],Lothar de Maizière,[Private_school__p012],"Huguenot, Private school"
106,qwen3:1.7b,Who halted the advance of Hulagu Khan across the Middle East?,"[the Mamluks of Egypt, Mamluks]",Frederick William,[Huguenot__p043],"Geology, Huguenot"
107,qwen3:1.7b,What halftime performer previously headlined Super Bowl XLVII?,[Beyoncé],<unknown>,[],"American Broadcasting Company, University of Chicago"
109,qwen3:1.7b,What is the name of the service that gets local businesses contract chances with the Super Bowl?,[Business Connect],Super Bowl,[American_Broadcasting_Company__p044],American Broadcasting Company
110,qwen3:1.7b,"According to Wilson's theorem, what factorial must be divisible by n if some integer n > 4 is to...",[(n − 1)!],2n,[<doc_id>],Oxygen


### 5.4 有答案卻拒答的案例（A_dev，RAG）

In [10]:
false_abstain = df[(df.test_set == "A_dev") & (df["mode"] == "rag") & (df.abstained.astype(bool))]
false_abstain[["gen_model", "question", "gold_answers", "gold_in_context", "context_ids"]]

,gen_model,question,gold_answers,gold_in_context,context_ids
40,qwen3:1.7b,What was the name of the experiment that tested how life originated?,"[Miller–Urey experiment, Chicago Pile-1]",False,"[Oxygen__p011, Oxygen__p003, Oxygen__p004]"
134,qwen3:4b-instruct,What chess grandmaster is also a university alumni?,[Samuel Reshevsky],False,"[University_of_Chicago__p000, University_of_Chicago__p027, University_of_Chicago__p028]"
146,qwen3:4b-instruct,"What are the two ABC affiliates for Tampa, Florida?",[WFTS-TV and WWSB],False,"[American_Broadcasting_Company__p076, American_Broadcasting_Company__p095, American_Broadcasting..."
170,qwen3:4b-instruct,What was the name of the experiment that tested how life originated?,"[Miller–Urey experiment, Chicago Pile-1]",False,"[Oxygen__p011, Oxygen__p003, Oxygen__p004]"
176,qwen3:4b-instruct,Which British sculptor whose work include the Queen Victoria memorial in front of Buckingham Pal...,[Thomas Brock],False,"[Victoria_and_Albert_Museum__p000, Victoria_and_Albert_Museum__p049, Victoria_and_Albert_Museum_..."
264,qwen3.5:9b-q4_K_M,What chess grandmaster is also a university alumni?,[Samuel Reshevsky],False,"[University_of_Chicago__p000, University_of_Chicago__p027, University_of_Chicago__p028]"
276,qwen3.5:9b-q4_K_M,"What are the two ABC affiliates for Tampa, Florida?",[WFTS-TV and WWSB],False,"[American_Broadcasting_Company__p076, American_Broadcasting_Company__p095, American_Broadcasting..."
300,qwen3.5:9b-q4_K_M,What was the name of the experiment that tested how life originated?,"[Miller–Urey experiment, Chicago Pile-1]",False,"[Oxygen__p011, Oxygen__p003, Oxygen__p004]"
306,qwen3.5:9b-q4_K_M,Which British sculptor whose work include the Queen Victoria memorial in front of Buckingham Pal...,[Thomas Brock],False,"[Victoria_and_Albert_Museum__p000, Victoria_and_Albert_Museum__p049, Victoria_and_Albert_Museum_..."


## 6. 怎麼解讀

- **隔離測試全部 PASS**：切換 Collection 不會撈到另一套的文件，這是企畫書 Stage 1 的驗收項目。
- **5.1 兩行都是 True**：三個模型用的是同一份索引，換生成模型不需要重建向量。
- **B_on_A 拒答率**：這就是「無答案題」的基準線。Stage 3 改成混合檢索後要再跑一次同樣的題目，比較拒答率有沒有下降（下降就是副作用）。
- **5.2 的模型比較**：大模型不一定比較好。可能答對率較高，卻比較常亂答或不照格式。只跑一次、題數又少，差異要謹慎解讀。
- 若某個模型在 4 節被「略過」，代表 Ollama 沒有那個模型名稱或呼叫失敗，錯誤訊息會印在上面。